In [29]:
import numpy as np
import pandas as pd


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, recall_score, precision_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.base import clone


DATA_PATH = "../data/Obesity.csv"
RANDOM_STATE = 150
TEST_SIZE = 0.20

df = pd.read_csv(DATA_PATH)

In [30]:
df.head(10)

,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II
5,Male,29.0,1.62,53.0,no,yes,2.0,3.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Automobile,Normal_Weight
6,Female,23.0,1.50,55.0,yes,yes,3.0,3.0,Sometimes,no,2.0,no,1.0,0.0,Sometimes,Motorbike,Normal_Weight
7,Male,22.0,1.64,53.0,no,no,2.0,3.0,Sometimes,no,2.0,no,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
8,Male,24.0,1.78,64.0,yes,yes,3.0,3.0,Sometimes,no,2.0,no,1.0,1.0,Frequently,Public_Transportation,Normal_Weight
9,Male,22.0,1.72,68.0,yes,yes,2.0,3.0,Sometimes,no,2.0,no,1.0,1.0,no,Public_Transportation,Normal_Weight


### Funções

#### Renomear Colunas

In [31]:
# -------------------------
# Função responsável por renomear colunas
# -------------------------
def renomear_colunas(df: pd.DataFrame) -> pd.DataFrame:
    colunas_renomeadas = {
        "Gender": "genero",
        "Age": "idade",
        "Height": "altura",
        "Weight": "peso",
        "family_history": "historico_familiar",
        "FAVC": "frequencia_alimentos_caloricos",
        "FCVC": "frequencia_consumo_vegetais",
        "NCP": "numero_refeicoes_por_dia",
        "CAEC": "consumo_lanches_entre_refeicoes",
        "SMOKE": "fumante",
        "CH2O": "consumo_agua_diario",
        "SCC": "monitora_ingestao_calorica",
        "FAF": "frequencia_semanal_atividade_fisica",
        "TUE": "tempo_uso_eletronicos",
        "CALC": "consumo_bebidas_alcoolicas",
        "MTRANS": "meio_transporte_habitual",
        "Obesity": "obesidade"
    }
    df = df.rename(columns=colunas_renomeadas)
    return df

#### Arredondar/Limitar Valores Ordinais

In [32]:
# -------------------------
# Função responsável por arredondar e limitar variáveis ordinais com ruído decimal
# Ex: frequencia_consumo_vegetais, numero_refeicoes_por_dia, consumo_agua_diario, frequencia_semanal_atividade_fisica, tempo_uso_eletronicos
# possuem valores decimais que não fazem sentido
# A ideia aqui é arredondar esses valores e limitar aos valores possíveis
# -------------------------
def arrendondar_limitar_valores_ordinais(df: pd.DataFrame) -> pd.DataFrame:
    """
    Arredonda e limita variáveis ordinais com ruído decimal:
        frequencia_consumo_vegetais: 1..3
        numero_refeicoes_por_dia: 1..4
        consumo_agua_diario: 1..3
        frequencia_semanal_atividade_fisica: 0..3 
        tempo_uso_eletronicos: 0..2
    """
    df = df.copy()

    for c in ["frequencia_consumo_vegetais", "numero_refeicoes_por_dia",
              "consumo_agua_diario", "frequencia_semanal_atividade_fisica", 
              "tempo_uso_eletronicos"]:
        df[c] = np.rint(df[c]).astype(int)

    df["frequencia_consumo_vegetais"]          = df["frequencia_consumo_vegetais"].clip(1, 3)
    df["numero_refeicoes_por_dia"]             = df["numero_refeicoes_por_dia"].clip(1, 4)
    df["consumo_agua_diario"]                  = df["consumo_agua_diario"].clip(1, 3)
    df["frequencia_semanal_atividade_fisica"]  = df["frequencia_semanal_atividade_fisica"].clip(0, 3)
    df["tempo_uso_eletronicos"]                = df["tempo_uso_eletronicos"].clip(0, 2)

    # criar variável ordinal para family_history
    df["historico_familiar_bin"] = df["historico_familiar"].map({"no": 0,"yes": 1})

    return df


#### Normalizar Valores

In [33]:
def normalizar_valores(df: pd.DataFrame) -> pd.DataFrame:
    
    # Converter para variável numérica (binária)
    df["monitora_ingestao_calorica_bin"]     = (df["monitora_ingestao_calorica"] == "yes").astype(int)
    df["frequencia_alimentos_caloricos_bin"] = (df["frequencia_alimentos_caloricos"] == "yes").astype(int)
    df["fumante_bin"]                        = (df["fumante"] == "yes").astype(int)

    bebidas = df["consumo_bebidas_alcoolicas"].map({
        "no": 0,
        "Sometimes": 1,
        "Frequently": 2,
        "Always": 3
    }).fillna(1).astype(int)
    
    lanches = df["consumo_lanches_entre_refeicoes"].map({
        "no": 0,
        "Sometimes": 1,
        "Frequently": 2,
        "Always": 3
    }).fillna(0).astype(int)

    # Normalizações 0..1
    df["frequencia_consumo_vegetais_n"]         = (df["frequencia_consumo_vegetais"] - 1) / 2
    df["consumo_agua_diario_n"]                 = (df["consumo_agua_diario"] - 1) / 2
    df["frequencia_semanal_atividade_fisica_n"] = df["frequencia_semanal_atividade_fisica"] / 3
    df["numero_refeicoes_por_dia_n"]            = (df["numero_refeicoes_por_dia"] - 1) / 3
    df["consumo_bebidas_alcoolicas_n"]          = bebidas / 3
    df["tempo_uso_eletronicos_n"]               = df["tempo_uso_eletronicos"] / 2
    df["consumo_lanches_entre_refeicoes_n"]     = lanches / 3

    return df

#### Criar Score de Ingestão de Calorias

In [34]:
# -------------------------
# Existe uma variável que captura a frequencia de consumo de alimentos calóricos (FAVC)
# Porém, apenas essa variável por si só não determina se um indivíduo será obeso ou não.
# É necessário considerar outros fatores comportamentais que influenciam a ingestão calórica total
# 
# Essa função é responsável por criar um score de ingestão calórica baseado em variáveis comportamentais, combinando:
#   - hábitos alimentares,
#   - padrão de refeições,
#   - consumo de vegetais (fator protetor),
#   - histórico familiar,
#   - nível de atividade física.
# Consumo de calorias está diretamente relacionado ao comportamento alimentar do indivíduo, 
# sendo um dos principais fatores para o desenvolvimento da obesidade.
# -------------------------
def criar_score_ingestao_calorias(df: pd.DataFrame) -> pd.DataFrame:
    """
    Proxy de carga calórica com base em:
    - frequencia_alimentos_caloricos (yes/no)
    - consumo_lanches_entre_refeicoes (no/Sometimes/Frequently/Always)
    - número de refeições por dia (1..4)
    - frequencia_consumo_vegetais (proteção)
    """
    df = df.copy()

    # Índice linear de ingestão calórica
    df["index_ingestao_calorica"] = (
        # Como a ingestão calórica é fortemente impactada pelo consumo frequente de alimentos calóricos,
        # esse é o principal fator, recebendo um peso maior na composição do índice
        1.2 * df["frequencia_alimentos_caloricos_bin"]

        # Comer entre refeições também impacta significativamente a ingestão calórica total
        # Por isso ele é o segundo fator mais importante
        + 0.9 * df["consumo_lanches_entre_refeicoes_n"]

        # Número de refeições pode influenciar a ingestão calórica total, mas tem um peso menor pois de acordo com as explorações
        # anteriores, o número de refeições não tem uma correlação tão forte com obesidade
        # Essa variável é um fator contextualizador para o cálculo do índice, por isso tem um peso menor
        + 0.4 * df["numero_refeicoes_por_dia_n"]

        # Consumo de vegetais tem efeito protetor, reduzindo a ingestão calórica total consumida nas refeições
        - 0.8 * df["frequencia_consumo_vegetais_n"]
    )

    # Sigmoide para score 0..1
    df["score_ingestao_calorica"] = 1 / (1 + np.exp(-df["index_ingestao_calorica"]))

    # A ideia aqui é capturar o efeito potencializado do histórico familiar (geneticamente predisposto) com a ingestão calórica 
    df["risco_genetico"] = df["historico_familiar_bin"] * df["score_ingestao_calorica"]

    # Balancear ingestão calórica com nível de atividade física
    df["balanco_caloria_atividade"] = df["score_ingestao_calorica"] / (df["frequencia_semanal_atividade_fisica"] + 1)

    return df

#### Criar Score de Perfil de Controle

In [35]:
# -------------------------
# Função responsável por criar um score de "perfil controlado/metabólico"
# A ideia é capturar o quão saudável/metabólico é o perfil do indivíduo com base em fatores comportamentais
# A combinação desses fatores pode indicar um perfil mais controlado/metabólico, reduzindo o risco de obesidade
# -------------------------
def criar_score_controle(df: pd.DataFrame) -> pd.DataFrame:
    """
    Score de "perfil controlado/metabólico" (0..1) com base em:
    + monitora_ingestao_calorica
    + frequencia_consumo_vegetais
    + consumo_agua_diario
    + frequencia_semanal_atividade_fisica
    + numero_refeicoes_por_dia
    - consumo_bebidas_alcoolicas
    - fumante
    - tempo_uso_eletronicos (sedentarismo)
    """
    df = df.copy()

    # Pesos heurísticos (monotônicos)
    df["score_controle_index"] = (
          1.5 * df["monitora_ingestao_calorica_bin"]
        + 1.2 * df["frequencia_consumo_vegetais_n"]
        + 1.0 * df["consumo_agua_diario_n"]
        + 1.3 * df["frequencia_semanal_atividade_fisica_n"]
        + 0.6 * df["numero_refeicoes_por_dia_n"]
        - 1.2 * df["consumo_bebidas_alcoolicas_n"]
        - 0.3 * df["fumante_bin"]
        - 0.3 * df["tempo_uso_eletronicos_n"]
    )

    # Normalizar 0..1
    df["score_controle"] = 1 / (1 + np.exp(-df["score_controle_index"]))

    return df

Criar Feature de Comportamentos

In [36]:
def criar_features_comportamento(df: pd.DataFrame) -> pd.DataFrame:
    """
    Criação de features para capturar interações entre variáveis comportamentais
    relevantes para obesidade, sem usar antropometria (peso, altura, IMC, etc)

    Ideia central
    -------------
      - Riscos costumam aparecer em combinação de fatores e não em variáveis isoladas
      - Por isso, criar features que capturem essas interações pode ajudar o modelo a identificar padrões mais complexos

    Cuidados
    --------
        - Evitar criar muitas features que possam levar a overfitting
        - Focar em interações que façam sentido do ponto de vista comportamental e de saúde
        - Colunas precisam ser normalizadas previamente (0..1) para quem os cálculos sejam na mesma escala ou não explodam em magnitude
    """
    df = df.copy()

    # Inatividade física 
    #   Pega-se a freq. semanal de atividade fisica normalizada e inverte-se
    #   Esse é uma das principais features, já que freq. de atividade física é a variável com maior correlação com obesidade,
    #   de acordo com a Análise Exploratória de Dados realizada no arquivo exploiration.ipynb
    df["inatividade"] = 1 - df["frequencia_semanal_atividade_fisica_n"]

    # Identificar quem é mais ativo em relação ao tipo de maio de transporte (caminhada ou bicicleta)
    # De acordo com a exploração, esse grupo tem menor prevalência de obesidade 
    #   Walking: 5.4%
    #   Bike: 14.3%
    #   Motorbike: 27.3%
    #   Automobile: 45.1%
    #   Public_Transportation: 48.0%
    df["transporte_passivo"] = df["meio_transporte_habitual"].isin({"Motorbike", "Automobile", "Public_Transportation"}).astype(int)

    # Sedentarismo: média entre inatividade física e uso de transporte passivo
    #   Esse score tenta capturar uma rotina mais passiva/sedentária da pessoa, combinando esses dois fatores
    #   A média foi escolhida para balancear os dois aspectos igualmente, mantendo o score entre 0 e 1
    #   Interpretação: 
    #    0   = mais ativo (atividade física frequente e transporte ativo) 
    #    0.5 = um fator ruim e o outro bom
    #    1   = mais sedentário (atividade física rara e transporte passivo)
    df["risco_sedentarismo"] = (df["inatividade"] + df["transporte_passivo"]) / 2
    
    # Alta ingestão calórica combinada com risco de sedentarismo
    #   Essa interação cresce quando há uma alta ingestão calórica e uma rotina mais sedentária
    #   A combinação de "comer muito e mover-se pouco" é mais informativa do que olhar para uma variável isoladamente
    df["ingestao_x_sedentarismo"] = df["index_ingestao_calorica"] * df["risco_sedentarismo"]
    
    # Score de controle/metabólico combinado com sedentarismo
    #   A ideia aqui é capturar o efeito potencializado de um perfil comportamental menos saudável (baixo score de controle) com uma rotina sedentária
    #   Essa interação pode indicar um risco aumentado de obesidade, já que tanto o perfil comportamental quanto o sedentarismo contribuem negativamente
    df["score_controle_x_sedentarismo"] = df["score_controle"] * df["risco_sedentarismo"]

    # Histórico familiar combinado com inatividade física
    #   Pré-disposição genética pode se manifestar com mais força em um estilo de vida mais sedentário
    #   Se historico_familiar_bin = 0, essa feature zera
    df["genetica_x_inatividade"] = df["historico_familiar_bin"] * df["inatividade"]

    # Ingestão calórica X Inatividade
    #   Feature que valida diretamente a associação entre uma alta ingestão calórica com a inatividade isoladamente (sem misturar transporte)
    df["ingestao_x_inatividade"] = df["index_ingestao_calorica"] * df["inatividade"]

    # Tempo de uso eletronico x Inatividade
    #   Caso o tempo de tela seja alto e a inatividade também, isso pode potencializar o sedentarismo, por mais que o TUE esteja mais relacionado com
    #   a idade do que de fato a falta de atividade
    df["tempo_eletronico_x_inatividade"] = df["tempo_uso_eletronicos_n"] * df["inatividade"]

    # Tempo de uso eletronico x Risco Sedentarismo
    #   Similar ao anterior, porém validando o TUE com o Risco de sedentarismo
    df["tempo_eletronico_x_inatividade"] = df["tempo_uso_eletronicos_n"] * df["risco_sedentarismo"]

    return df

#### Criar Target

In [37]:
def criar_target_binario(df: pd.DataFrame) -> pd.Series:
    """
    Cria um target binário com base na coluna 'obesidade'.
    """
    return df["obesidade"].isin({"Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III"}).astype(int)

#### Construir Pipeline

In [38]:
def build_pipeline(X: pd.DataFrame, model, dense_for_model: bool = False) -> Pipeline:
    """
    Constrói um Pipeline do scikit-learn com:
      1) Pré-processamento (numéricas + categóricas)
      2) Modelo (estimador sklearn)

    Parâmetros
    ----------
    X : pd.DataFrame
        DataFrame contendo APENAS as features (sem o alvo). É usado aqui para:
        - descobrir automaticamente quais colunas são numéricas vs categóricas
        - garantir que o ColumnTransformer saiba exatamente o que transformar
    model :
        Um estimador do scikit-learn (ex: LogisticRegression, LinearSVC, RandomForest, XGBoost wrapper, etc)
        Ele será plugado como o último passo do pipeline.
    dense_for_model : bool, default=False
        Controla se a saída do OneHotEncoder será:
        - esparsa (sparse) -> eficiente em memória para muitas categorias
        - densa (dense)    -> necessária para alguns modelos/implementações que não aceitam sparse

        Regra prática:
        - Modelos lineares do sklearn normalmente aceitam sparse => deixe False.
        - Alguns algoritmos / integrações podem exigir dense => use True.

    Retorno
    -------
    Pipeline
        Pipeline com dois steps:
        - "prep": transformações de dados (numéricas + categóricas)
        - "model": modelo treinável
    """

    # ---------------
    # Identificar colunas categóricas e numéricas
    # ---------------
    # filtra o tipo objeto para achar as categóricas
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

    # qualquer outra coluna diferente das filtradas acima, entra como numérica
    num_cols = [c for c in X.columns if c not in cat_cols]

    # Construir o pré-processamento, onde colunas numéricas são escaladas e colunas categóricas são one-hot encoded, já que ainda
    # existem columnas que não foram normalizadas no dataset original
    preprocess = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            # OneHotEncoder gera matrizes muito densas; sparse economiza memória
            # Porém, alguns modelos não aceitam sparse (ex: HistGradientBoosting do sklearn),
            # por isso o parâmetro dense_for_model para focar o dense
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=not dense_for_model), cat_cols), 
        ],
        remainder="drop",
    )

    return Pipeline(steps=[("prep", preprocess), ("model", model)])


### Execução e escolha do modelo

In [39]:
# Pré-processamento conforme dicionário
df = renomear_colunas(df)
df = arrendondar_limitar_valores_ordinais(df)
df = normalizar_valores(df)

# Feature engineering 100% comportamental, sem levar em conta variáveis antropométricas como IMC, peso, altura
df = criar_score_ingestao_calorias(df)
df = criar_score_controle(df)
df = criar_features_comportamento(df)

with pd.option_context('display.max_columns', None):
    display(df.head())

,genero,idade,altura,peso,historico_familiar,frequencia_alimentos_caloricos,frequencia_consumo_vegetais,numero_refeicoes_por_dia,consumo_lanches_entre_refeicoes,fumante,consumo_agua_diario,monitora_ingestao_calorica,frequencia_semanal_atividade_fisica,tempo_uso_eletronicos,consumo_bebidas_alcoolicas,meio_transporte_habitual,obesidade,historico_familiar_bin,monitora_ingestao_calorica_bin,frequencia_alimentos_caloricos_bin,fumante_bin,frequencia_consumo_vegetais_n,consumo_agua_diario_n,frequencia_semanal_atividade_fisica_n,numero_refeicoes_por_dia_n,consumo_bebidas_alcoolicas_n,tempo_uso_eletronicos_n,consumo_lanches_entre_refeicoes_n,index_ingestao_calorica,score_ingestao_calorica,risco_genetico,balanco_caloria_atividade,score_controle_index,score_controle,inatividade,transporte_passivo,risco_sedentarismo,ingestao_x_sedentarismo,score_controle_x_sedentarismo,genetica_x_inatividade,ingestao_x_inatividade,tempo_eletronico_x_inatividade
0,Female,21.0,1.62,64.0,yes,no,2,3,Sometimes,no,2,no,0,1,no,Public_Transportation,Normal_Weight,1,0,0,0,0.5,0.5,0.000000,0.666667,0.000000,0.5,0.333333,0.166667,0.541570,0.54157,0.541570,1.350000,0.794130,1.000000,1,1.000000,0.166667,0.794130,1.000000,0.166667,0.500000
1,Female,21.0,1.52,56.0,yes,no,3,3,Sometimes,yes,3,yes,3,0,Sometimes,Public_Transportation,Normal_Weight,1,1,0,1,1.0,1.0,1.000000,0.666667,0.333333,0.0,0.333333,-0.233333,0.441930,0.44193,0.110482,4.700000,0.990987,0.000000,1,0.500000,-0.116667,0.495493,0.000000,-0.000000,0.000000
2,Male,23.0,1.80,77.0,yes,no,2,3,Sometimes,no,2,no,2,1,Frequently,Public_Transportation,Normal_Weight,1,0,0,0,0.5,0.5,0.666667,0.666667,0.666667,0.5,0.333333,0.166667,0.541570,0.54157,0.180523,1.416667,0.804815,0.333333,1,0.666667,0.111111,0.536544,0.333333,0.055556,0.333333
3,Male,27.0,1.80,87.0,no,no,3,3,Sometimes,no,2,no,2,0,Frequently,Walking,Overweight_Level_I,0,0,0,0,1.0,0.5,0.666667,0.666667,0.666667,0.0,0.333333,-0.233333,0.441930,0.00000,0.147310,2.166667,0.897216,0.333333,0,0.166667,-0.038889,0.149536,0.000000,-0.077778,0.000000
4,Male,22.0,1.78,89.8,no,no,2,1,Sometimes,no,2,no,0,0,Sometimes,Public_Transportation,Overweight_Level_II,0,0,0,0,0.5,0.5,0.000000,0.000000,0.333333,0.0,0.333333,-0.100000,0.475021,0.00000,0.475021,0.700000,0.668188,1.000000,1,1.000000,-0.100000,0.668188,0.000000,-0.100000,0.000000


In [40]:
df.columns

Index(['genero', 'idade', 'altura', 'peso', 'historico_familiar',
       'frequencia_alimentos_caloricos', 'frequencia_consumo_vegetais',
       'numero_refeicoes_por_dia', 'consumo_lanches_entre_refeicoes',
       'fumante', 'consumo_agua_diario', 'monitora_ingestao_calorica',
       'frequencia_semanal_atividade_fisica', 'tempo_uso_eletronicos',
       'consumo_bebidas_alcoolicas', 'meio_transporte_habitual', 'obesidade',
       'historico_familiar_bin', 'monitora_ingestao_calorica_bin',
       'frequencia_alimentos_caloricos_bin', 'fumante_bin',
       'frequencia_consumo_vegetais_n', 'consumo_agua_diario_n',
       'frequencia_semanal_atividade_fisica_n', 'numero_refeicoes_por_dia_n',
       'consumo_bebidas_alcoolicas_n', 'tempo_uso_eletronicos_n',
       'consumo_lanches_entre_refeicoes_n', 'index_ingestao_calorica',
       'score_ingestao_calorica', 'risco_genetico',
       'balanco_caloria_atividade', 'score_controle_index', 'score_controle',
       'inatividade', 'transporte_p

In [41]:
# Target
y = criar_target_binario(df)
y.head(5)

0    0
1    0
2    0
3    0
4    0
Name: obesidade, dtype: int64

In [42]:
# Remover target e antropometria totalmente (sem usar nem como latente)
# drop_cols = ["Obesity", "Weight", "Height", "TUE"]
# X = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Escolha das features, removendo o target e algumas variável antropométricas
# além disso, também foram removidas algumas features redundantes, para uma melhor calibragem do modelo
features = [
    'genero',
    'idade',
    # 'altura',                                 <---- antropométrico
    # 'peso',                                   <---- antropométrico
    # 'historico_familiar',
    # 'frequencia_alimentos_caloricos',
    # 'frequencia_consumo_vegetais',
    # 'numero_refeicoes_por_dia',
    # 'consumo_lanches_entre_refeicoes',
    # 'fumante',
    # 'consumo_agua_diario',
    # 'monitora_ingestao_calorica',
    # 'frequencia_semanal_atividade_fisica',
    # 'tempo_uso_eletronicos',
    # 'consumo_bebidas_alcoolicas',
    # 'meio_transporte_habitual',
    # 'obesidade',                              <---- target
    'historico_familiar_bin',
    'monitora_ingestao_calorica_bin',
    'frequencia_alimentos_caloricos_bin',
    'fumante_bin',
    # 'consumo_bebidas_alcoolicas_num',
    'frequencia_consumo_vegetais_n',
    'consumo_agua_diario_n',
    'frequencia_semanal_atividade_fisica_n',
    'numero_refeicoes_por_dia_n',
    'consumo_bebidas_alcoolicas_n',
    'tempo_uso_eletronicos_n',
    'consumo_lanches_entre_refeicoes_n',
    'index_ingestao_calorica',
    # 'score_ingestao_calorica',
    'risco_genetico',
    'balanco_caloria_atividade',
    'score_controle_index',
    'score_controle',
    'inatividade',
    'transporte_passivo',
    'risco_sedentarismo',
    'ingestao_x_sedentarismo',
    'score_controle_x_sedentarismo',
    'genetica_x_inatividade',
    'ingestao_x_inatividade',
    'tempo_eletronico_x_inatividade'
]

X = df[features]

X.head(5)

,genero,idade,historico_familiar_bin,monitora_ingestao_calorica_bin,frequencia_alimentos_caloricos_bin,fumante_bin,frequencia_consumo_vegetais_n,consumo_agua_diario_n,frequencia_semanal_atividade_fisica_n,numero_refeicoes_por_dia_n,...,score_controle_index,score_controle,inatividade,transporte_passivo,risco_sedentarismo,ingestao_x_sedentarismo,score_controle_x_sedentarismo,genetica_x_inatividade,ingestao_x_inatividade,tempo_eletronico_x_inatividade
0,Female,21.0,1,0,0,0,0.5,0.5,0.000000,0.666667,...,1.350000,0.794130,1.000000,1,1.000000,0.166667,0.794130,1.000000,0.166667,0.500000
1,Female,21.0,1,1,0,1,1.0,1.0,1.000000,0.666667,...,4.700000,0.990987,0.000000,1,0.500000,-0.116667,0.495493,0.000000,-0.000000,0.000000
2,Male,23.0,1,0,0,0,0.5,0.5,0.666667,0.666667,...,1.416667,0.804815,0.333333,1,0.666667,0.111111,0.536544,0.333333,0.055556,0.333333
3,Male,27.0,0,0,0,0,1.0,0.5,0.666667,0.666667,...,2.166667,0.897216,0.333333,0,0.166667,-0.038889,0.149536,0.000000,-0.077778,0.000000
4,Male,22.0,0,0,0,0,0.5,0.5,0.000000,0.000000,...,0.700000,0.668188,1.000000,1,1.000000,-0.100000,0.668188,0.000000,-0.100000,0.000000


In [43]:
# Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Teste de alguns modelos
candidatos = {
    # esparso OK
    "LogReg": (LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE), False),
    "LinearSVC": (LinearSVC(class_weight="balanced", random_state=RANDOM_STATE), False),
    "SGD_logloss": (SGDClassifier(loss="log_loss", class_weight="balanced", random_state=RANDOM_STATE), False),

    # denso recomendado
    "RandomForest": (RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, class_weight="balanced_subsample"), True),
    "ExtraTrees": (ExtraTreesClassifier(n_estimators=800, random_state=RANDOM_STATE, class_weight="balanced_subsample"), True),
    "HistGB": (HistGradientBoostingClassifier(random_state=RANDOM_STATE), True),
}

resultados = []

for nome, (modelo, dense_flag) in candidatos.items():
    pipe = build_pipeline(X_train, model=modelo, dense_for_model=dense_flag)
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)

    # ROC-AUC: tenta proba, senão decision_function
    if hasattr(pipe, "predict_proba"):
        score_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    else:
        score_auc = roc_auc_score(y_test, pipe.decision_function(X_test))

    acc = accuracy_score(y_test, pred)

    resultados.append((nome, acc, score_auc))

    print(f"\n=== {nome} ===")
    print(f"Acurácia: {acc:.4f} | ROC-AUC: {score_auc:.4f}")
    print("Matriz de confusão:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))

resultados



=== LogReg ===
Acurácia: 0.7896 | ROC-AUC: 0.8576
Matriz de confusão:
 [[157  71]
 [ 18 177]]
              precision    recall  f1-score   support

           0       0.90      0.69      0.78       228
           1       0.71      0.91      0.80       195

    accuracy                           0.79       423
   macro avg       0.81      0.80      0.79       423
weighted avg       0.81      0.79      0.79       423


=== LinearSVC ===
Acurácia: 0.7849 | ROC-AUC: 0.8549
Matriz de confusão:
 [[155  73]
 [ 18 177]]
              precision    recall  f1-score   support

           0       0.90      0.68      0.77       228
           1       0.71      0.91      0.80       195

    accuracy                           0.78       423
   macro avg       0.80      0.79      0.78       423
weighted avg       0.81      0.78      0.78       423


=== SGD_logloss ===
Acurácia: 0.7116 | ROC-AUC: 0.7654
Matriz de confusão:
 [[157  71]
 [ 51 144]]
              precision    recall  f1-score   support

[('LogReg', 0.789598108747045, 0.8576135852451641),
 ('LinearSVC', 0.7848699763593381, 0.8548920377867746),
 ('SGD_logloss', 0.7115839243498818, 0.765395861448493),
 ('RandomForest', 0.8817966903073287, 0.9590418353576249),
 ('ExtraTrees', 0.8747044917257684, 0.9450404858299595),
 ('HistGB', 0.9196217494089834, 0.9611673414304993)]

---

#### ⭐️ Escolha do modelo

Com base no resultado acima, o HistGB parece ser o melhor candidato, com uma acurácia acima dos 91%, ROC-AUC de 0.96 e recall da classe 1 de 0.90 e precision de 0.93.

---

### Validação das features

Verificar quais features tem mais peso e se é possível melhorar, ou quem sabe, apenas simplificar as features

In [44]:
# -------------------------
# Análise de coeficientes das features
# -------------------------

def importance(pipe, X_test, y_test, scoring="roc_auc", n_jobs=-1, n_repeats=20, random_state=150) -> pd.DataFrame:
    r = permutation_importance(
        pipe,
        X_test, y_test,
        n_repeats=n_repeats,
        random_state=random_state,
        scoring=scoring,
        n_jobs=n_jobs
    )

    return pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": r.importances_mean,
        "importance_std": r.importances_std,
        "abs_importance": np.abs(r.importances_mean),
    }).sort_values("abs_importance", ascending=False)

(modelo, dense_flag) = candidatos["HistGB"]
final_pipe = build_pipeline(X_train, model=modelo, dense_for_model=dense_flag)
final_pipe.fit(X_train, y_train)

# feature_names = final_pipe.named_steps["prep"].get_feature_names_out()

df = importance(final_pipe, X_test, y_test)

df



,feature,importance_mean,importance_std,abs_importance
1,idade,0.126605,0.012104,0.126605
14,risco_genetico,0.093161,0.013796,0.093161
0,genero,0.039245,0.004284,0.039245
12,consumo_lanches_entre_refeicoes_n,0.021268,0.005905,0.021268
9,numero_refeicoes_por_dia_n,0.009704,0.004047,0.009704
13,index_ingestao_calorica,0.009398,0.002564,0.009398
21,ingestao_x_sedentarismo,0.007645,0.002523,0.007645
7,consumo_agua_diario_n,0.006416,0.001149,0.006416
22,score_controle_x_sedentarismo,0.006370,0.001848,0.006370
10,consumo_bebidas_alcoolicas_n,0.005494,0.002271,0.005494


In [45]:
df = importance(final_pipe, X_test, y_test, "recall")
df

,feature,importance_mean,importance_std,abs_importance
1,idade,0.280256,0.023516,0.280256
14,risco_genetico,0.215385,0.025227,0.215385
0,genero,0.137179,0.010240,0.137179
12,consumo_lanches_entre_refeicoes_n,0.073846,0.017941,0.073846
21,ingestao_x_sedentarismo,0.066923,0.015595,0.066923
13,index_ingestao_calorica,0.054359,0.013115,0.054359
17,score_controle,0.026923,0.009848,0.026923
15,balanco_caloria_atividade,0.025897,0.008804,0.025897
22,score_controle_x_sedentarismo,0.025641,0.011467,0.025641
9,numero_refeicoes_por_dia_n,0.022564,0.012499,0.022564


##### Análise sobre os resultados

- 4 features se destacam em AUC e recall: idade, risco_genetico, genero, consumo_lanches_entre_refeicoes_n, sendo bem maiores do que as restantes
  - idade, risco_genetico, genero ajudam a identificar quem o sujeito é
  - mesmo sem idade/altura/IMC, o modelo é movido a demografia/predisposição
  - vale o teste para ver como o modelo se comportará sem essas 3 features demográficas
  
- Algumas features sobem bastante no recall, quando comparado ao AUC. Pelo visto, essas features ajudam a reduzir falsos negativos -> ajudando a capturar mais obesos
  - score_controle: 0.0025 (AUC) vs 0.0269 (recall)
  - balanco_caloria_atividade: 0.0006 (AUC) vs 0.0259 (recall)
  - tempo_uso_eletronicos_n: 0.0011 (AUC) vs 0.0218 (recall)
  - ingestao_x_sedentarismo: 0.0076 (AUC) vs 0.0669 (recall)
  - ingestao_x_inatividade: 0 (AUC) vs 0.0100 (recall)
  - tempo_eletronico_x_inatividade: 0.0042 (AUC) vs 0.0177 (recall)
  
- Algumas features geralmente "não são confiáveis" quando a mean é <= do que o std. Daqui a pouco valeria realizar um novo teste sem eles.
  - AUC: frequencia_semanal_atividade_fisica_n e ingestao_x_inatividade
  - Recall: frequencia_consumo_vegetais_n, fumante_bin, consumo_agua_diario_n

- Existem algumas features que realmente parecem não estarem sendo usadas e há grandes indícios de removê-las.
  - inatividade
  - historico_familiar_bin
  - frequencia_alimentos_caloricos_bin
  - frequencia_semanal_atividade_fisica_n

In [46]:
# ---------
# Primeiro teste: remover as variáveis demográfica e comparar os resultados
# 
# Resultado: piora em tudo ❌
# ---------

X_test_1 = X.copy()

remover = ["genero", "idade", "risco_genetico", "historico_familiar_bin", "genetica_x_inatividade"]
X_test_1 = X_test_1.drop(columns=[c for c in remover if c in X_test_1.columns])

X_train_test_1, X_test_test_1, y_train_test_1, y_test_test_1 = train_test_split(
    X_test_1, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

(modelo, dense_flag) = candidatos["HistGB"]
pipe_test_1 = build_pipeline(X_train_test_1, model=modelo, dense_for_model=dense_flag)
pipe_test_1.fit(X_train_test_1, y_train_test_1)
pred_test_1 = pipe_test_1.predict(X_test_test_1)
proba_test_1 = pipe_test_1.predict_proba(X_test_test_1)[:, 1]

score_auc = roc_auc_score(y_test_test_1, proba_test_1)
acc = accuracy_score(y_test_test_1, pred_test_1)

print(f"\n=== HistGB (sem genero, idade, risco_generico) ===")
print(f"Acurácia: {acc:.4f} | ROC-AUC: {score_auc:.4f}")
print("Matriz de confusão:\n", confusion_matrix(y_test_test_1, pred_test_1))
print(classification_report(y_test_test_1, pred_test_1))

print(importance(pipe_test_1, X_test_test_1, y_test_test_1))
print(importance(pipe_test_1, X_test_test_1, y_test_test_1, "recall"))


=== HistGB (sem genero, idade, risco_generico) ===
Acurácia: 0.7967 | ROC-AUC: 0.8668
Matriz de confusão:
 [[186  42]
 [ 44 151]]
              precision    recall  f1-score   support

           0       0.81      0.82      0.81       228
           1       0.78      0.77      0.78       195

    accuracy                           0.80       423
   macro avg       0.80      0.80      0.80       423
weighted avg       0.80      0.80      0.80       423

                                  feature  importance_mean  importance_std  \
9       consumo_lanches_entre_refeicoes_n         0.100567        0.016923   
17                ingestao_x_sedentarismo         0.036713        0.009278   
12                   score_controle_index         0.030759        0.009459   
6              numero_refeicoes_por_dia_n         0.021478        0.005885   
7            consumo_bebidas_alcoolicas_n         0.016596        0.002164   
10                index_ingestao_calorica         0.016349        0.004577

In [47]:
# ---------
# Segundo teste: remover as features não confiáveis ou sem importância
#
# Resultado: Manteve os mesmos resultados. Removerei essas três features ✅
# ---------

X_test_2 = X.copy()

remover = ["inatividade", "frequencia_alimentos_caloricos_bin","frequencia_semanal_atividade_fisica_n"]
X_test_2 = X_test_2.drop(columns=[c for c in remover if c in X_test_2.columns])

X_train_test_2, X_test_test_2, y_train_test_2, y_test_test_2 = train_test_split(
    X_test_2, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

(modelo, dense_flag) = candidatos["HistGB"]
pipe_test_2 = build_pipeline(X_train_test_2, model=modelo, dense_for_model=dense_flag)
pipe_test_2.fit(X_train_test_2, y_train_test_2)
pred_test_2 = pipe_test_2.predict(X_test_test_2)
proba_test_2 = pipe_test_2.predict_proba(X_test_test_2)[:, 1]

score_auc = roc_auc_score(y_test_test_2, proba_test_2)
acc = accuracy_score(y_test_test_2, pred_test_2)

print(f"\n=== HistGB (sem genero, idade, risco_generico) ===")
print(f"Acurácia: {acc:.4f} | ROC-AUC: {score_auc:.4f}")
print("Matriz de confusão:\n", confusion_matrix(y_test_test_2, pred_test_2))
print(classification_report(y_test_test_2, pred_test_2))

print(importance(pipe_test_2, X_test_test_2, y_test_test_2))
print(importance(pipe_test_2, X_test_test_2, y_test_test_2, "recall"))


=== HistGB (sem genero, idade, risco_generico) ===
Acurácia: 0.9196 | ROC-AUC: 0.9612
Matriz de confusão:
 [[214  14]
 [ 20 175]]
              precision    recall  f1-score   support

           0       0.91      0.94      0.93       228
           1       0.93      0.90      0.91       195

    accuracy                           0.92       423
   macro avg       0.92      0.92      0.92       423
weighted avg       0.92      0.92      0.92       423

                              feature  importance_mean  importance_std  \
1                               idade         0.126593        0.012094   
12                     risco_genetico         0.093129        0.013799   
0                              genero         0.039232        0.004283   
10  consumo_lanches_entre_refeicoes_n         0.021254        0.005898   
7          numero_refeicoes_por_dia_n         0.009703        0.004045   
11            index_ingestao_calorica         0.009392        0.002508   
18            ingestao_x

##### Decisão
- Removerei mais 3 features "inatividade", "frequencia_alimentos_caloricos_bin","frequencia_semanal_atividade_fisica_n
- Manterei features como "genero", "idade", "risco_genetico", "historico_familiar_bin", "genetica_x_inatividade" pois elas ajudam a equilibrar o modelo

### Validação do modelo em sub-grupos
- A ideia é tentar ver se o modelo tem algum viés e corrigí-lo

In [48]:
def _gerar_auc(y_true, y_score):
    """
        AUC só existe se tiver as duas classes (0 e 1)
        Se um subgrupo tiver só 0s ou só 1s, o AUC não é definido e o sklearn lança erro
        Essa função tem como objetivo garantir que o relatório adicione o AUC escore ou NaN
    """
    if len(pd.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)

def rodar_validacao_por_subgrupo(X_test: pd.DataFrame, y_test, proba, pred) -> pd.DataFrame:

    # Indexa y_test, proba e pred nas mesmas linhas de X_test, evitando problemas de índices após shuffle ou split
    X_test = X_test.copy()
    y_test = pd.Series(y_test, index=X_test.index)
    proba  = pd.Series(proba, index=X_test.index)
    pred   = pd.Series(pred, index=X_test.index)

    linhas = []

    def adicionar_linha(grupo, subgrupo, mask):
        y_g = y_test[mask]
        p_g = proba[mask]
        pred_g = pred[mask]

        # labels=[0, 1] => garante a ordem correta, mesmo se num subgrupo não aparecer alguma classe na predição
        # ravel() -> transforma a matriz 2x2 em 4 números, nessa ordem:
        #  - vn = verdadeiro/negativo
        #  - fp = falso/positivo
        #  - fn = falso/negativo
        #  - vp = verdadeiro/positivo
        vn, fp, fn, vp = confusion_matrix(y_g, pred_g, labels=[0, 1]).ravel()

        linhas.append({

            # Tamanho e Prevalência
            "grupo": grupo,
            "subgrupo": subgrupo,
            "n": int(mask.sum()), # numero de linhas do subgrupo
            "pos_n": int((y_g == 1).sum()), # quantidade de positivos reais (obeso y=1) do subgrupo
            "neg_n": int((y_g == 0).sum()), # quantidade de negativos reais (obeso y=0) do subgrupo
            "pos_rate": float((y_g == 1).mean()), # prevalencia de obesidade naquele subgrupo. Isso é importante pois a acc pode enganar em subgrupos raros. Ex: Se pos_rate é 3%, prever sempre 0 dá 97% de acc
            
            # Probabilidade
            "auc": _gerar_auc(y_g, p_g), 

            # Decisão (threshold)
            "recall_1": recall_score(y_g, pred_g, zero_division=0), # dos obesos reais, quantos o modelo pegou
            "precision_1": precision_score(y_g, pred_g, zero_division=0), # dos que o modelo marcou como obeso, quantos eram obesos
            "accuracy": accuracy_score(y_g, pred_g), # acurácia de acerto
            
            # Matriz de confusão
            "vn": int(vn), "fp": int(fp), "fn": int(fn), "vp": int(vp), 
        })

    # Global
    adicionar_linha("Tudo", "Tudo", pd.Series(True, index=X_test.index))

    # Gênero
    if "genero" in X_test.columns:
        adicionar_linha("genero", "Female", (X_test["genero"] == "Female"))
        adicionar_linha("genero", "Male",   (X_test["genero"] == "Male"))

    # Idade
    if "idade" in X_test.columns:
        adicionar_linha("idade", "<=30", (X_test["idade"] <= 30))
        adicionar_linha("idade", ">30",  (X_test["idade"] > 30))

    # Histórico familiar
    if "historico_familiar_bin" in X_test.columns:
        adicionar_linha("historico_familiar_bin", "0", (X_test["historico_familiar_bin"] == 0))
        adicionar_linha("historico_familiar_bin", "1", (X_test["historico_familiar_bin"] == 1))

    return pd.DataFrame(linhas)

report = rodar_validacao_por_subgrupo(X_test_test_2, y_test_test_2, proba_test_2, pred_test_2)

print(report.to_string(index=False))

                 grupo subgrupo   n  pos_n  neg_n  pos_rate      auc  recall_1  precision_1  accuracy  vn  fp  fn  vp
                  Tudo     Tudo 423    195    228  0.460993 0.961167  0.897436     0.925926  0.919622 214  14  20 175
                genero   Female 225    105    120  0.466667 0.969683  0.923810     0.932692  0.933333 113   7   8  97
                genero     Male 198     90    108  0.454545 0.948611  0.866667     0.917647  0.904040 101   7  12  78
                 idade     <=30 351    160    191  0.455840 0.969224  0.912500     0.948052  0.937322 183   8  14 146
                 idade      >30  72     35     37  0.486111 0.902703  0.828571     0.828571  0.833333  31   6   6  29
historico_familiar_bin        0  85      3     82  0.035294 0.808943  0.000000     0.000000  0.964706  82   0   3   0
historico_familiar_bin        1 338    192    146  0.568047 0.954641  0.911458     0.925926  0.908284 132  14  17 175


#### Insights

- Tem mais mulheres do que homens, pouca coisa, mas tem. Isso faz com que os indices do homem sejam levemente piores.
  - tentar talvez um reweighting por genero x classe
  - threshold escolhido para maximizar o pior recall entre os generos

- Subgrupo de idade >30 é muito menor do que <=30. Isso pode treinar o modelo de forma equivocada
  - posso talvez usar sample_weight para dar mais peso a idade>30
  - e/ou posso tentar garantir mais exemplos de >30 nos folds

- Da mesma forma, o subgrupo de historico_familiar_bin=0 é muito menor do que o outro, tendo recall e precision = 0.
  - pos_n=3 vs neg_n=82 !!! Poucos casos reais onde não há histórico familiar e a pessoa seja obesa
  - será que os dados de treino estão com problemas? Validar isso


#### 🔎 Análise sobre historico_familiar_bin
- Entender como os dados estão distribuídos
- E se a gente removesse apenas as variáveis relacionadas a historico_familiar?

In [49]:
# ------------
# Validar quantidade de obesos vs historico_familiar_bin
#
# Resultado: Dados de treino possuem quase nenhum exemplo de obeso onde o historico_familiar_bin=0. Isso pode estar atrapalhando o desempenho do modelo
# ------------
y_s = pd.Series(y_train_test_2, index=X_train_test_2.index, name="y")
df = pd.DataFrame({
    "historico_familiar_bin": X_train_test_2["historico_familiar_bin"].values,
    "y": y_s.values
}, index=X_train_test_2.index)

tab = pd.crosstab(df["historico_familiar_bin"], df["y"], margins=True)
tab.columns = [f"y={c}" for c in tab.columns]

print(tab)

                        y=0  y=1  y=All
historico_familiar_bin                 
0                       295    5    300
1                       616  772   1388
All                     911  777   1688


In [50]:
# ---------
# Terceiro teste: E se eu removesse apenas as features relacionadas a histórico_familiar?
# ---------

X_test_3 = X_test_2.copy()

remover = ["risco_genetico", "historico_familiar_bin", "genetica_x_inatividade"]
X_test_3 = X_test_3.drop(columns=[c for c in remover if c in X_test_3.columns])

X_train_test_3, X_test_test_3, y_train_test_3, y_test_test_3 = train_test_split(
    X_test_3, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

(modelo, dense_flag) = candidatos["HistGB"]
pipe_test_3 = build_pipeline(X_train_test_3, model=modelo, dense_for_model=dense_flag)
pipe_test_3.fit(X_train_test_3, y_train_test_3)
pred_test_3 = pipe_test_3.predict(X_test_test_3)
proba_test_3 = pipe_test_3.predict_proba(X_test_test_3)[:, 1]

score_auc = roc_auc_score(y_test_test_3, proba_test_3)
acc = accuracy_score(y_test_test_3, pred_test_3)

print(f"\n=== HistGB (risco_genetico, historico_familiar_bin, genetica_x_inatividade) ===")
print(f"Acurácia: {acc:.4f} | ROC-AUC: {score_auc:.4f}")
print("Matriz de confusão:\n", confusion_matrix(y_test_test_3, pred_test_3))
print(classification_report(y_test_test_3, pred_test_3))

print(importance(pipe_test_3, X_test_test_3, y_test_test_3))
print(importance(pipe_test_3, X_test_test_3, y_test_test_3, "recall"))


=== HistGB (risco_genetico, historico_familiar_bin, genetica_x_inatividade) ===
Acurácia: 0.9078 | ROC-AUC: 0.9543
Matriz de confusão:
 [[214  14]
 [ 25 170]]
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       228
           1       0.92      0.87      0.90       195

    accuracy                           0.91       423
   macro avg       0.91      0.91      0.91       423
weighted avg       0.91      0.91      0.91       423

                              feature  importance_mean  importance_std  \
1                               idade         0.185453        0.014916   
9   consumo_lanches_entre_refeicoes_n         0.049893        0.008861   
0                              genero         0.037284        0.004295   
6          numero_refeicoes_por_dia_n         0.019931        0.006852   
16            ingestao_x_sedentarismo         0.018636        0.003756   
17      score_controle_x_sedentarismo         0.012379        0.00284

In [51]:
report_test_3 = rodar_validacao_por_subgrupo(X_test_test_3, y_test_test_3, proba_test_3, pred_test_3)

print(report_test_3.to_string(index=False))

 grupo subgrupo   n  pos_n  neg_n  pos_rate      auc  recall_1  precision_1  accuracy  vn  fp  fn  vp
  Tudo     Tudo 423    195    228  0.460993 0.954307  0.871795     0.923913  0.907801 214  14  25 170
genero   Female 225    105    120  0.466667 0.960079  0.885714     0.939394  0.920000 114   6  12  93
genero     Male 198     90    108  0.454545 0.945010  0.855556     0.905882  0.893939 100   8  13  77
 idade     <=30 351    160    191  0.455840 0.959964  0.875000     0.933333  0.914530 181  10  20 140
 idade      >30  72     35     37  0.486111 0.918147  0.857143     0.882353  0.875000  33   4   5  30


##### 💡 Insights:
Global:
  - AUC, Precision e ACC se mantiveram alto:
     - AUC: 0.9543 vs 0.9612
     - Precision: 0.9259 vs 0.9239
     - ACC: 0.9196 vs 0.9078
  - Recall foi o único que caiu mais: de 0.8974 (FN=20) para 0.8718 (FN=25)
  
Quando analisamos os sub-grupos, notamos que:
  - Gênero: Recall caiu em ambos male/female. O gap entre eles diminui devido a queda de 0.92 para 0.88 no female
  - Idade: 
    - No grupo <=30 observa AUC praticamente igual, mas queda no recall: 0.9125 vs 0.8750. 
    - Entretanto, percebemos uma boa melhora no AUC e recall no grupo >30:
      - Recall: 0.8286 -> 0.8571 
      - AUC: 0.9027 -> 0.9181

Aparentemente, as features de genética puxavam o grupo >30 para baixo.

Além disso, a remoção dessas features remove também um subgrupo que tinha prevalência baixíssima e recall=0.

#### ➡️ Experimento: Tentar melhorar threshold (sem historico_familiar_bin)
 - Tentar melhorar o recall em alguns grupos
 - Mas ainda sim mantendo um equilíbrio (um pouco mais conservador)
 - Usar F1 score para ajudar nisso

In [52]:
# ------------
# Testar vários thresholds 
# ------------

def validar_thresholds(
    X_test: pd.DataFrame,
    y_test,
    proba,
    thresholds=None,
    min_n: int = 50
) -> tuple[pd.DataFrame, dict]:
    """
    Executa rodar_validacao_por_subgrupo para vários thresholds e devolve:
      - df_resumo: 1 linha por threshold com métricas globais + fairness (recall e FPR)
      - reports_por_threshold: dict[threshold] -> DataFrame do report completo por subgrupo
    """

    # Indexa y_test e proba nas mesmas linhas de X_test, evitando problemas de índices após shuffle ou split
    X_test = X_test.copy()
    y_test = pd.Series(y_test, index=X_test.index).astype(int)
    proba  = pd.Series(proba, index=X_test.index).astype(float)

    # Cria uma lista de thresholds de 0.05 a 0.95 com 91 pontos
    # Ou seja: 0.05, 0.06, 0.07 ... 0.94, 0.95
    # Todos esses valores serão testados para encontrar o que tráz melhor resultado de F1 score
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 91)

    linhas = []
    reports_por_threshold = {}

    for t in thresholds:

        # A comparação com o threshold (proba >= t) gera um vetor booleano
        #   True => quando a probabilidade é maior ou igual a t
        #   False quando é menor que t
        #
        # .astype(int) converte o boleano para integer
        #
        # Ao diminuir t (ex: 0.3), mais pessoas viram 1, aumentando recall e pegando mais obesos, diminuindo Falsos/Negativos
        #   - Porém diminui precision...
        pred = (proba >= t).astype(int)

        report = rodar_validacao_por_subgrupo(X_test=X_test, y_test=y_test, proba=proba, pred=pred)

        # salva o report por threshold para poder buscá-lo no final, quando o melhor threshold for identificado
        reports_por_threshold[float(t)] = report

        # linha global
        linha_global = report[(report["grupo"] == "Tudo") & (report["subgrupo"] == "Tudo")].iloc[0]
        neg_global = int(linha_global["neg_n"])
        fpr_global = (int(linha_global["fp"]) / neg_global) if neg_global > 0 else np.nan # fpr = falso/positivo ratio

        # fairness só nos grupos escolhidos
        mask_fair = report["grupo"].isin(("genero", "idade"))
        sub = report[mask_fair].copy()

        # recall fairness
        pior_recall = float(sub["recall_1"].min()) if len(sub) else np.nan
        gap_recall  = float(sub["recall_1"].max() - sub["recall_1"].min()) if len(sub) else np.nan

        # FPR fairness (FP / NEG) por subgrupo
        if len(sub):
            sub["fpr"] = sub.apply(lambda r: (r["fp"] / r["neg_n"]) if r["neg_n"] > 0 else np.nan,axis=1)
            pior_fpr = float(sub["fpr"].max())  # pior = maior taxa de falso positivo
            gap_fpr  = float(sub["fpr"].max() - sub["fpr"].min())
        else:
            pior_fpr = np.nan
            gap_fpr = np.nan

        # F1 global (mais "balanceado" que só recall/precision)
        prec_g = float(linha_global["precision_1"])
        rec_g  = float(linha_global["recall_1"])
        f1_global = (2 * prec_g * rec_g / (prec_g + rec_g)) if (prec_g + rec_g) > 0 else 0.0

        linhas.append({
            "threshold": float(t),

            # métricas globais
            "auc_global": float(linha_global["auc"]) if pd.notna(linha_global["auc"]) else np.nan,
            "recall_global": rec_g,
            "precision_global": prec_g,
            "f1_global": float(f1_global),
            "accuracy_global": float(linha_global["accuracy"]),
            "fp_global": int(linha_global["fp"]),
            "fn_global": int(linha_global["fn"]),
            "fpr_global": float(fpr_global),
            "pred_pos_rate": float(pred.mean()),

            # fairness entre subgrupos
            "pior_recall_subgrupos": pior_recall,
            "gap_recall_subgrupos": gap_recall,
            "pior_fpr_subgrupos": pior_fpr,
            "gap_fpr_subgrupos": gap_fpr,
        })

    df_resumo = pd.DataFrame(linhas).sort_values("threshold").reset_index(drop=True)
    return df_resumo, reports_por_threshold


# -----------------------
# Hora de rodar a função
# -----------------------
df_thr, reports_por_thr = validar_thresholds(X_test=X_test_test_3, y_test=y_test_test_3, proba=proba_test_3)

print(df_thr.head(10).to_string(index=False))
print("")
print(reports_por_thr)

 threshold  auc_global  recall_global  precision_global  f1_global  accuracy_global  fp_global  fn_global  fpr_global  pred_pos_rate  pior_recall_subgrupos  gap_recall_subgrupos  pior_fpr_subgrupos  gap_fpr_subgrupos
      0.05    0.954307       0.974359          0.681004   0.801688         0.777778         89          5    0.390351       0.659574               0.961905              0.026984            0.783784           0.483784
      0.06    0.954307       0.974359          0.690909   0.808511         0.787234         85          5    0.372807       0.650118               0.961905              0.026984            0.729730           0.438063
      0.07    0.954307       0.969231          0.710526   0.819957         0.803783         77          6    0.337719       0.628842               0.952381              0.036508            0.702703           0.436036
      0.08    0.954307       0.969231          0.721374   0.827133         0.813239         73          6    0.320175       0.619385

In [53]:
# -----------------------
# Achar o melhor thrshold
# -----------------------
def escolher_threshold_balanceado(
    df_thr: pd.DataFrame,
    recall_min_subgrupos: float = 0.85,
    gap_recall_max: float = 0.08,
    fpr_global_max: float = 0.35,
    pior_fpr_max: float = 0.45,
    gap_fpr_max: float = 0.20
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Escolhe o melhor threshold com base em:
      - constraints de fairness (recall + FPR)
      - objetivo: maximizar F1 global
    
     Variáveis de controle:
       1) Garantir fairness mínima em recall (não deixar ninguém muito pior)
       2) Limitar taxa de falso positivo (FPR) global e por subgrupo
       3) Dentro disso, maximizar F1 global (balanceia precision e recall)

    recall_min_subgrupos = 0.85
    gap_recall_max = 0.08

    fpr_global_max = 0.35        - ajuste conforme tolerância a FP (0.35 é um bom começo)
    pior_fpr_max = 0.45          - limite do pior subgrupo (ex.: idade >30 não pode explodir)
    gap_fpr_max = 0.20           - evita FPR muito desigual
    """

    # Filtrar do df_thr para encontrar apenas entradas dentro dos valores acima
    candidatos = df_thr[
        (df_thr["pior_recall_subgrupos"] >= recall_min_subgrupos) &
        (df_thr["gap_recall_subgrupos"] <= gap_recall_max) &
        (df_thr["fpr_global"] <= fpr_global_max) &
        (df_thr["pior_fpr_subgrupos"] <= pior_fpr_max) &
        (df_thr["gap_fpr_subgrupos"] <= gap_fpr_max)
    ].copy()

    # Caso alguém retorne no filtro acima, orderná-los por f1, precision e thrshold
    if len(candidatos) > 0:
        melhor = candidatos.sort_values(
            ["f1_global", "precision_global", "threshold"],
            ascending=[False, False, True]
        ).head(1)

    else:
        # fallback 1: maximiza F1 global com fairness mínima de recall
        fallback = df_thr[
            (df_thr["pior_recall_subgrupos"] >= recall_min_subgrupos) &
            (df_thr["gap_recall_subgrupos"] <= gap_recall_max)
        ].copy()

        if len(fallback) > 0:
            melhor = fallback.sort_values(
                ["f1_global", "precision_global", "threshold"],
                ascending=[False, False, True]
            ).head(1)
        
        else:

            # fallback 2: maximiza F1 global sem constraints
            melhor = df_thr.sort_values(
                ["f1_global", "precision_global", "threshold"],
                ascending=[False, False, True]
            ).head(1)
    
    return (melhor, candidatos)


melhor, candidatos = escolher_threshold_balanceado(df_thr)

thr_escolhido = float(melhor["threshold"].iloc[0])
print("Threshold escolhido:", thr_escolhido)
print(melhor.to_string(index=False))

report_escolhido = reports_por_thr[thr_escolhido]
print("\n=== Report por subgrupo no threshold escolhido ===")
print(report_escolhido.to_string(index=False))

print("\n=== Report anterior por subgrupo (Comparação) ===")
print(report_test_3.to_string(index=False))

# Ver os 10 melhores candidatos para inspecionar trade-offs
print("\n=== Top 10 candidatos (ordenado por F1) ===")
print(
    candidatos.sort_values(["f1_global", "threshold"], ascending=[False, True])
    .head(10)
    .to_string(index=False)
    if len(candidatos) else
    "Nenhum candidato passou nas constraints de FPR + recall. Ajuste os limites."
)

Threshold escolhido: 0.4599999999999999
 threshold  auc_global  recall_global  precision_global  f1_global  accuracy_global  fp_global  fn_global  fpr_global  pred_pos_rate  pior_recall_subgrupos  gap_recall_subgrupos  pior_fpr_subgrupos  gap_fpr_subgrupos
      0.46    0.954307       0.882051          0.914894   0.898172         0.907801         16         23    0.070175       0.444444               0.866667              0.028571            0.135135           0.077544

=== Report por subgrupo no threshold escolhido ===
 grupo subgrupo   n  pos_n  neg_n  pos_rate      auc  recall_1  precision_1  accuracy  vn  fp  fn  vp
  Tudo     Tudo 423    195    228  0.460993 0.954307  0.882051     0.914894  0.907801 212  16  23 172
genero   Female 225    105    120  0.466667 0.960079  0.895238     0.930693  0.920000 113   7  11  94
genero     Male 198     90    108  0.454545 0.945010  0.866667     0.896552  0.893939  99   9  12  78
 idade     <=30 351    160    191  0.455840 0.959964  0.881250    

In [54]:

# ---------------------------------------
# Validar melhor threshold por CV
# ---------------------------------------

def selecionar_threshold_por_cv(
    pipeline,
    X: pd.DataFrame,
    y,
    n_splits: int = 5,
    random_state: int = RANDOM_STATE
) -> dict:
    """
    Faz CV e escolhe threshold em cada fold (usando validar_thresholds + escolher_threshold_balanceado).
    Retorna um dicionário com:
      - thresholds_por_fold
      - threshold_final (mediana)
      - resumo_por_fold (métricas do threshold escolhido em cada fold)
    """

    X = X.copy()
    y = pd.Series(y, index=X.index).astype(int)
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    thresholds_por_fold = []
    resumo_por_fold = []

    for fold, (idx_treino, idx_valid) in enumerate(skf.split(X, y), start=1):

        X_treino = X.iloc[idx_treino].copy()
        y_treino = y.iloc[idx_treino].copy()
        X_valid  = X.iloc[idx_valid].copy()
        y_valid  = y.iloc[idx_valid].copy()

        # Treina o pipeline no treino do fold
        pipeline_fold = clone(pipeline)
        pipeline_fold.fit(X_treino, y_treino)

        # Probabilidades no valid do fold
        proba_valid = pipeline_fold.predict_proba(X_valid)[:, 1]

        # Testa todos thresholds (mesmo que anterior,entre 0.05, 0.06 .. 0.94, 0.95)
        df_thr, reports_por_thr = validar_thresholds(X_test=X_valid, y_test=y_valid, proba=proba_valid)

        # Escolhe threshold balanceado no fold
        thr_fold, candidatos = escolher_threshold_balanceado(df_thr=df_thr)
        
        thr_fold = float(thr_fold["threshold"].iloc[0])

        # Salva o valor do threshold do fold em thresholds_por_fold
        thresholds_por_fold.append(thr_fold)

        # Pega métricas globais no threshold escolhido
        linha = df_thr[df_thr["threshold"] == thr_fold].iloc[0].to_dict()
        linha["fold"] = fold
        linha["n_valid"] = int(len(X_valid))

        resumo_por_fold.append(linha)

        print(f"[Fold {fold}] threshold={thr_fold:.2f} | "
              f"recall={linha['recall_global']:.3f} | "
              f"precision={linha['precision_global']:.3f} | "
              f"f1={linha['f1_global']:.3f} | "
              f"pior_recall_sub={linha['pior_recall_subgrupos']:.3f} | "
              f"pior_fpr_sub={linha['pior_fpr_subgrupos']:.3f}")

    df_resumo = pd.DataFrame(resumo_por_fold).sort_values("fold").reset_index(drop=True)

    # Threshold final (mediana é robusta a outliers)
    threshold_final = float(np.median(thresholds_por_fold))

    return {
        "thresholds_por_fold": thresholds_por_fold,
        "threshold_final": threshold_final,
        "resumo_por_fold": df_resumo
    }

resultado_cv = selecionar_threshold_por_cv(
    pipeline=pipe_test_3,
    X=X_train_test_3,
    y=y_train_test_3,
    n_splits=5,
    random_state=RANDOM_STATE
)

print("\nThresholds por fold:", resultado_cv["thresholds_por_fold"])
print("Threshold final (mediana):", resultado_cv["threshold_final"])

print("\n=== Resumo por fold ===")
print(resultado_cv["resumo_por_fold"].to_string(index=False))

[Fold 1] threshold=0.46 | recall=0.890 | precision=0.852 | f1=0.871 | pior_recall_sub=0.857 | pior_fpr_sub=0.217
[Fold 2] threshold=0.39 | recall=0.910 | precision=0.840 | f1=0.874 | pior_recall_sub=0.892 | pior_fpr_sub=0.286
[Fold 3] threshold=0.40 | recall=0.910 | precision=0.861 | f1=0.885 | pior_recall_sub=0.887 | pior_fpr_sub=0.333
[Fold 4] threshold=0.36 | recall=0.884 | precision=0.820 | f1=0.851 | pior_recall_sub=0.855 | pior_fpr_sub=0.183
[Fold 5] threshold=0.30 | recall=0.903 | precision=0.805 | f1=0.851 | pior_recall_sub=0.885 | pior_fpr_sub=0.455

Thresholds por fold: [0.4599999999999999, 0.38999999999999996, 0.3999999999999999, 0.35999999999999993, 0.3]
Threshold final (mediana): 0.38999999999999996

=== Resumo por fold ===
 threshold  auc_global  recall_global  precision_global  f1_global  accuracy_global  fp_global  fn_global  fpr_global  pred_pos_rate  pior_recall_subgrupos  gap_recall_subgrupos  pior_fpr_subgrupos  gap_fpr_subgrupos  fold  n_valid
      0.46    0.94325

In [55]:

# Aplicar o threshold contra os subgrupos para compará-los
thr_final = resultado_cv["threshold_final"]

proba_test_3_s = pd.Series(proba_test_3, index=X_test_test_3.index)
pred_test = (proba_test_3_s >= thr_final).astype(int)

report_teste = rodar_validacao_por_subgrupo(
    X_test=X_test_test_3,
    y_test=y_test_test_3,
    proba=proba_test_3_s,
    pred=pred_test
)

print(f"\n=== Report por subgrupo (threshold CV: {resultado_cv['threshold_final']}) ===")
print(report_teste.to_string(index=False))

report_escolhido = reports_por_thr[thr_escolhido]
print(f"\n=== Report por subgrupo no threshold escolhido: {thr_escolhido} ===")
print(report_escolhido.to_string(index=False))

print("\n=== Report por subgrupo (padrão 0.5) ===")
print(report_test_3.to_string(index=False))


=== Report por subgrupo (threshold CV: 0.38999999999999996) ===
 grupo subgrupo   n  pos_n  neg_n  pos_rate      auc  recall_1  precision_1  accuracy  vn  fp  fn  vp
  Tudo     Tudo 423    195    228  0.460993 0.954307  0.897436     0.888325  0.900709 206  22  20 175
genero   Female 225    105    120  0.466667 0.960079  0.914286     0.905660  0.915556 110  10   9  96
genero     Male 198     90    108  0.454545 0.945010  0.877778     0.868132  0.883838  96  12  11  79
 idade     <=30 351    160    191  0.455840 0.959964  0.900000     0.911392  0.914530 177  14  16 144
 idade      >30  72     35     37  0.486111 0.918147  0.885714     0.794872  0.833333  29   8   4  31

=== Report por subgrupo no threshold escolhido: 0.4599999999999999 ===
 grupo subgrupo   n  pos_n  neg_n  pos_rate      auc  recall_1  precision_1  accuracy  vn  fp  fn  vp
  Tudo     Tudo 423    195    228  0.460993 0.954307  0.882051     0.914894  0.907801 212  16  23 172
genero   Female 225    105    120  0.466667 0.9

### ✅ Escolha final

#### 💡 Insight: Comparação dos três thresholds

##### Métricas Globais

| Threshold | Recall | Precision | F1 | Accuracy | FP | FN | VP | VN |
|-----------|--------|-----------|-------|----------|----|----|----|----|
| **0.39 (CV)** | 89.7% | 88.8% | 89.3% | 90.1% | 22 | 20 | 175 | 206 |
| **0.46 (escolhido)** | 88.2% | 91.5% | 89.8% | 90.8% | 16 | 23 | 172 | 212 |
| **0.50 (padrão)** | 87.2% | 92.4% | 89.7% | 90.8% | 14 | 25 | 170 | 214 |

##### Métricas por Gênero

| Threshold | Subgrupo | Recall | Precision | Accuracy | FP | FN |
|-----------|----------|--------|-----------|----------|----|----|
| **0.39** | Female | 91.4% | 90.6% | 91.6% | 10 | 9 |
| **0.39** | Male | 87.8% | 86.8% | 88.4% | 12 | 11 |
| **0.46** | Female | 89.5% | 93.1% | 92.0% | 7 | 11 |
| **0.46** | Male | 86.7% | 89.7% | 89.4% | 9 | 12 |
| **0.50** | Female | 88.6% | 93.9% | 92.0% | 6 | 12 |
| **0.50** | Male | 85.6% | 90.6% | 89.4% | 8 | 13 |

##### Métricas por Idade

| Threshold | Subgrupo | Recall | Precision | Accuracy | FP | FN |
|-----------|----------|--------|-----------|----------|----|----|
| **0.39** | ≤30 | 90.0% | 91.1% | 91.5% | 14 | 16 |
| **0.39** | >30 | 88.6% | 79.5% | 83.3% | 8 | 4 |
| **0.46** | ≤30 | 88.1% | 92.8% | 91.5% | 11 | 19 |
| **0.46** | >30 | 88.6% | 86.1% | 87.5% | 5 | 4 |
| **0.50** | ≤30 | 87.5% | 93.3% | 91.5% | 10 | 20 |
| **0.50** | >30 | 85.7% | 88.2% | 87.5% | 4 | 5 |

##### Análise de Fairness

| Métrica | 0.39 (CV) | 0.46 | 0.50 |
|---------|-----------|------|------|
| **Gap Recall (Gênero)** | 3.6 pp | 2.9 pp | 3.0 pp |
| **Gap Recall (Idade)** | 1.4 pp | 0.5 pp | 1.8 pp |
| **Pior Recall** | 87.8% (Male) | 86.7% (Male) | 85.6% (Male) |
| **Pior Precision** | 79.5% (>30) | 86.1% (>30) | 88.2% (>30) |

##### 🎯 Decisão

**Threshold 0.46** parece ser o melhor equilíbrio:
- Melhor F1 global (89.8%)
- Melhor precision (91.5%) com recall ainda alto (88.2%)
- Menos falsos positivos (16 vs 22 do 0.39)
- Menor gap de recall entre idades (0.5pp)
- Precision no grupo >30 muito melhor (86.1% vs 79.5%)

**Threshold 0.39** é melhor se:
- Prioridade absoluta for recall (capturar mais casos positivos)
- Custo de falsos negativos for muito alto

**Threshold 0.50** é melhor se:
- Precision for crítica (evitar falsos alarmes)
- Custo de falsos positivos for muito alto